# LDM Stage 2 Training - Kaggle Notebook

**Latent Diffusion Model (Stage 2) Baseline Training**

This notebook trains the LDM Stage 2 baseline using the frozen VQ autoencoder from Experiment 1.

## Prerequisites
1. **GPU**: Enable T4 or P100 GPU in Kaggle settings (16GB recommended)
2. **Datasets to Attach** (in Kaggle UI → Add Data):
   - Thermal images dataset: `new-dataset/Processed`
   - VQ model dataset: Upload your VQ checkpoint + latent stats as a Kaggle dataset
     - `best_model (1).pth` (VQ checkpoint)
     - `vq_latent_statistics.yaml` (latent statistics)
3. **GitHub Repo**: Your code repository (will be cloned/pulled in the notebook)

## Training Options
- **Trial Run**: 5 epochs (smoke test, ~30 minutes)
- **Full Training**: 100 epochs with early stopping (~6-10 hours)

## How It Works
- **Code**: Pulled from GitHub (always latest version)
- **Data**: Attached as Kaggle input datasets
- **Checkpoints**: Auto-uploaded to persistent Kaggle dataset

## Expected Outputs
- Trained diffusion model checkpoint
- Generated sample images
- Training logs and metrics
- Checkpoints persisted to Kaggle Dataset

---

## 🔧 Configuration

**IMPORTANT: Update these values before running!**

In [ ]:
# ============================================================================
# USER CONFIGURATION - UPDATE THESE!
# ============================================================================

# Your Kaggle username (for checkpoint dataset persistence)
KAGGLE_USERNAME = "YOUR_USERNAME_HERE"  # ⚠️ UPDATE THIS!

# GitHub repository URL (for code)
GITHUB_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ⚠️ UPDATE THIS!

# Training mode: 'trial' for 5 epochs, 'full' for 100 epochs
TRAINING_MODE = "trial"  # Options: 'trial' or 'full'

# Kaggle Input Datasets (attach these in Kaggle UI)
THERMAL_DATASET_PATH = "/kaggle/input/new-dataset/Processed/No_mixed"  # Thermal images
VQ_CHECKPOINT_PATH = "/kaggle/input/vq-model/best_model (1).pth"  # VQ checkpoint (attach as dataset)
LATENT_STATS_PATH = "/kaggle/input/vq-model/vq_latent_statistics.yaml"  # Latent stats (attach as dataset)

# Checkpoint dataset name (will be created automatically)
CHECKPOINT_DATASET_NAME = f"{KAGGLE_USERNAME}/ldm-stage2-checkpoints"

# ============================================================================

print("Configuration:")
print(f"  Kaggle Username: {KAGGLE_USERNAME}")
print(f"  GitHub URL: {GITHUB_URL}")
print(f"  Training Mode: {TRAINING_MODE}")
print(f"  Checkpoint Dataset: {CHECKPOINT_DATASET_NAME}")

if KAGGLE_USERNAME == "YOUR_USERNAME_HERE" or "YOUR_USERNAME" in GITHUB_URL:
    print("\n⚠️  WARNING: Please update KAGGLE_USERNAME and GITHUB_URL before running!")

## 📦 Environment Setup

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
import torch
import numpy as np
import yaml
from PIL import Image
import matplotlib.pyplot as plt

# Detect Kaggle environment
IS_KAGGLE = Path('/kaggle/input').exists()
print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Current directory: {os.getcwd()}")

# Clone or pull repository from GitHub
if IS_KAGGLE:
    REPO_NAME = 'Mixed-Pixels'
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ⚠️ UPDATE THIS!
    
    REPO_PATH = Path('/kaggle/working') / REPO_NAME
    
    if not REPO_PATH.exists():
        print(f"\n📦 Cloning repository from {GITHUB_URL}...")
        !git clone {GITHUB_URL} /kaggle/working/{REPO_NAME}
        print("   ✅ Repository cloned")
    else:
        print(f"\n🔄 Repository exists, pulling latest changes...")
        !cd /kaggle/working/{REPO_NAME} && git reset --hard HEAD && git pull
        print("   ✅ Repository updated")
    
    os.chdir(REPO_PATH)
    WORK_DIR = REPO_PATH
else:
    # Local: navigate to repo root
    repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    os.chdir(repo_root)
    WORK_DIR = Path.cwd()

print(f"Working directory: {WORK_DIR}")

# Add src to Python path
sys.path.insert(0, str(WORK_DIR / 'src'))

# Verify GPU
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Count: {torch.cuda.device_count()}")
else:
    print("⚠️  WARNING: No GPU detected!")

## 🔍 Verify Prerequisites

In [ ]:
print("Verifying prerequisites...\n")

# Check repository (code from GitHub)
if WORK_DIR.exists():
    print(f"✅ Repository cloned/pulled: {WORK_DIR}")
    src_path = WORK_DIR / 'src'
    if src_path.exists():
        print(f"   ✅ src/ directory found")
    else:
        print(f"   ❌ src/ directory missing!")
else:
    print(f"❌ Repository not found at {WORK_DIR}")

# Check thermal dataset (attached as Kaggle input)
thermal_path = Path(THERMAL_DATASET_PATH)
if thermal_path.exists():
    num_images = len(list(thermal_path.rglob('*.png'))) + len(list(thermal_path.rglob('*.jpg')))
    print(f"✅ Thermal dataset found: {num_images} images")
else:
    print(f"❌ Thermal dataset not found at {THERMAL_DATASET_PATH}")
    print(f"   Make sure to attach the thermal dataset in Kaggle!")

# Check VQ checkpoint (attached as Kaggle input)
vq_checkpoint_path = Path(VQ_CHECKPOINT_PATH)
if vq_checkpoint_path.exists():
    checkpoint_size = vq_checkpoint_path.stat().st_size / (1024**2)
    print(f"✅ VQ checkpoint found: {checkpoint_size:.2f} MB")
else:
    print(f"❌ VQ checkpoint not found at {VQ_CHECKPOINT_PATH}")
    print(f"   Make sure to attach the VQ model dataset in Kaggle!")

# Check latent statistics (attached as Kaggle input)
latent_stats_path = Path(LATENT_STATS_PATH)
if latent_stats_path.exists():
    with open(latent_stats_path) as f:
        stats = yaml.safe_load(f)
    print(f"✅ Latent statistics found")
    print(f"   Mean: {stats['quantized']['global_mean']:.6f}")
    print(f"   Std: {stats['quantized']['global_std']:.6f}")
else:
    print(f"❌ Latent statistics not found at {LATENT_STATS_PATH}")
    print(f"   Make sure the yaml file is in the VQ model dataset!")

print("\n" + "="*70)

## 📊 Load VQ Model and Statistics

In [ ]:
from models.vq_autoencoder import VQAutoencoder
from models.ldm.stage2.latent_scaler import LatentScaler

# Load VQ checkpoint
print("Loading frozen VQ autoencoder...")
vq_checkpoint = torch.load(VQ_CHECKPOINT_PATH, map_location='cpu')

# Create VQ model
vq_config = vq_checkpoint['config'] if 'config' in vq_checkpoint else {
    'model': {
        'in_channels': 3,
        'latent_channels': 4,
        'base_channels': 32,
        'channel_multipliers': [1, 2, 4, 4],
        'num_res_blocks': 2,
        'num_embeddings': 1024,
        'commitment_cost': 0.25
    }
}

vq_model = VQAutoencoder(
    in_channels=vq_config['model']['in_channels'],
    latent_channels=vq_config['model']['latent_channels'],
    base_channels=vq_config['model']['base_channels'],
    channel_multipliers=vq_config['model']['channel_multipliers'],
    num_res_blocks=vq_config['model']['num_res_blocks'],
    num_embeddings=vq_config['model']['num_embeddings'],
    commitment_cost=vq_config['model']['commitment_cost']
)

# Load weights (strict=True to catch any mismatch)
vq_model.load_state_dict(vq_checkpoint['model_state_dict'], strict=True)
vq_model.eval()

# Freeze all parameters
for param in vq_model.parameters():
    param.requires_grad = False

print(f"✅ VQ model loaded (epoch {vq_checkpoint['epoch']})")
print(f"   Parameters frozen: {sum(1 for p in vq_model.parameters() if not p.requires_grad)}")

# Load latent statistics
print("\nLoading latent statistics...")
with open(LATENT_STATS_PATH) as f:
    latent_stats = yaml.safe_load(f)

scaler = LatentScaler(
    mean=latent_stats['quantized']['global_mean'],
    std=latent_stats['quantized']['global_std']
)

print(f"✅ Latent scaler initialized")
print(f"   Mean: {scaler.mean:.6f}")
print(f"   Std: {scaler.std:.6f}")
print(f"   Scaling factor: {1/scaler.std:.2f}x")

## 🏗️ Initialize LDM Components

In [ ]:
from models.ldm.stage2.unet import DiffusionUNet
from models.ldm.stage2.noise_scheduler import NoiseScheduler
from models.ldm.stage2.ema import EMA

# Initialize U-Net
print("Initializing Diffusion U-Net...")
unet = DiffusionUNet(
    in_channels=4,
    out_channels=4,
    base_channels=64,  # Approved: 23.2M params
    channel_multipliers=[1, 2, 4, 4],
    num_res_blocks=2,
    attention_levels=[2, 3],
    dropout=0.0,
    time_emb_dim=256,
    num_groups=32
)

num_params = sum(p.numel() for p in unet.parameters())
print(f"✅ U-Net initialized: {num_params/1e6:.1f}M parameters")

# Initialize noise scheduler
print("\nInitializing noise scheduler...")
scheduler = NoiseScheduler(num_timesteps=1000, schedule='cosine')
print(f"✅ Noise scheduler initialized (cosine, T=1000)")

# Initialize EMA
print("\nInitializing EMA...")
ema = EMA(unet, decay=0.9999)
print(f"✅ EMA initialized (decay=0.9999)")

# Move models to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vq_model = vq_model.to(device)
unet = unet.to(device)
scheduler.to(device)

print(f"\n✅ All models moved to {device}")

## 📝 Training Configuration

In [ ]:
# Training hyperparameters based on mode
if TRAINING_MODE == "trial":
    MAX_EPOCHS = 5
    SAVE_EVERY = 2
    SAMPLE_EVERY = 2
    EARLY_STOPPING_PATIENCE = 10  # Won't trigger in 5 epochs
    print("🧪 TRIAL MODE: 5 epochs smoke test")
elif TRAINING_MODE == "full":
    MAX_EPOCHS = 100
    SAVE_EVERY = 5
    SAMPLE_EVERY = 10
    EARLY_STOPPING_PATIENCE = 10
    print("🚀 FULL TRAINING MODE: 100 epochs with early stopping")
else:
    raise ValueError(f"Invalid TRAINING_MODE: {TRAINING_MODE}. Use 'trial' or 'full'")

# Fixed hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
GRAD_CLIP = 1.0
NUM_WORKERS = 2

# Create output directories
OUTPUT_DIR = WORK_DIR / f"outputs/ldm_stage2_{TRAINING_MODE}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
SAMPLES_DIR = OUTPUT_DIR / "samples"
LOGS_DIR = OUTPUT_DIR / "logs"

for dir_path in [OUTPUT_DIR, CHECKPOINT_DIR, SAMPLES_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

print(f"\nTraining Configuration:")
print(f"  Max Epochs: {MAX_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Gradient Clip: {GRAD_CLIP}")
print(f"  Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"\nOutput Directories:")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"  Samples: {SAMPLES_DIR}")
print(f"  Logs: {LOGS_DIR}")

## 🎲 Setup Data Loaders

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets.thermal_dataset import ThermalImageDataset

# Image transforms
transform = transforms.Compose([
    transforms.Resize((480, 640)),
    transforms.ToTensor()
])

# Create datasets
print("Creating datasets...")
train_dataset = ThermalImageDataset(
    root_dir=THERMAL_DATASET_PATH,
    split='train',
    transform=transform,
    train_ratio=0.8,
    val_ratio=0.2,
    test_ratio=0.0,
    seed=42
)

val_dataset = ThermalImageDataset(
    root_dir=THERMAL_DATASET_PATH,
    split='val',
    transform=transform,
    train_ratio=0.8,
    val_ratio=0.2,
    test_ratio=0.0,
    seed=42
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"✅ Datasets created")
print(f"   Train: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"   Val: {len(val_dataset)} images ({len(val_loader)} batches)")

## 🔧 Initialize Optimizer and Tracking

In [ ]:
import torch.nn.functional as F
from torch.optim import Adam

# Optimizer
optimizer = Adam(unet.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.999))

# Training state
training_state = {
    'epoch': 0,
    'best_val_loss': float('inf'),
    'train_losses': [],
    'val_losses': [],
    'patience_counter': 0
}

print("✅ Optimizer initialized")
print(f"   Trainable parameters: {sum(p.numel() for p in unet.parameters() if p.requires_grad)/1e6:.1f}M")

## 📤 Checkpoint Persistence Functions

**These functions handle uploading checkpoints to Kaggle Dataset so they survive session crashes.**

In [ ]:
def upload_checkpoint_to_kaggle_dataset(checkpoint_dir, dataset_name):
    """
    Upload all checkpoints to Kaggle Dataset for persistence.
    
    Args:
        checkpoint_dir: Directory containing checkpoints
        dataset_name: Kaggle dataset name (e.g., 'username/dataset-name')
    """
    if not IS_KAGGLE:
        print("Not on Kaggle, skipping upload")
        return
    
    try:
        print(f"\n📤 Uploading checkpoints to {dataset_name}...")
        
        # Create temporary dataset directory
        dataset_dir = Path('/kaggle/temp_dataset')
        if dataset_dir.exists():
            shutil.rmtree(dataset_dir)
        dataset_dir.mkdir()
        
        # Copy all checkpoints
        checkpoint_path = Path(checkpoint_dir)
        if checkpoint_path.exists():
            for ckpt_file in checkpoint_path.glob('*.pth'):
                shutil.copy(ckpt_file, dataset_dir / ckpt_file.name)
                print(f"   Preparing: {ckpt_file.name}")
        
        # Create dataset metadata
        metadata = {
            "title": "Mixed Pixels LDM Stage 2 Checkpoints",
            "id": dataset_name,
            "licenses": [{"name": "CC0-1.0"}]
        }
        
        with open(dataset_dir / "dataset-metadata.json", "w") as f:
            json.dump(metadata, f)
        
        # Check if dataset exists
        check_cmd = f"kaggle datasets status {dataset_name}"
        result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
        
        if result.returncode == 0:
            # Dataset exists - create new version
            create_cmd = f"kaggle datasets version -p {dataset_dir} -m 'Updated checkpoints'"
            result = subprocess.run(create_cmd, shell=True, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"   ✅ Uploaded to Kaggle Dataset (version updated)")
            else:
                print(f"   ⚠️  Upload failed: {result.stderr}")
        else:
            # Dataset doesn't exist - create it
            create_cmd = f"kaggle datasets create -p {dataset_dir}"
            result = subprocess.run(create_cmd, shell=True, capture_output=True, text=True)
            if result.returncode == 0:
                print(f"   ✅ Created Kaggle Dataset: {dataset_name}")
            else:
                print(f"   ⚠️  Creation failed: {result.stderr}")
        
        # Cleanup
        shutil.rmtree(dataset_dir, ignore_errors=True)
        
    except Exception as e:
        print(f"   ⚠️  Checkpoint upload error: {e}")


def save_checkpoint(epoch, unet, ema, optimizer, train_losses, val_losses, best_val_loss, checkpoint_dir, filename):
    """
    Save training checkpoint.
    
    Args:
        epoch: Current epoch
        unet: U-Net model
        ema: EMA module
        optimizer: Optimizer
        train_losses: List of training losses
        val_losses: List of validation losses
        best_val_loss: Best validation loss so far
        checkpoint_dir: Directory to save checkpoint
        filename: Checkpoint filename
    """
    checkpoint_path = Path(checkpoint_dir) / filename
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': unet.state_dict(),
        'ema_state_dict': ema.shadow,
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'config': {
            'in_channels': 4,
            'out_channels': 4,
            'base_channels': 64,
            'channel_multipliers': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attention_levels': [2, 3],
            'num_timesteps': 1000,
            'schedule': 'cosine'
        },
        'latent_stats': {
            'mean': scaler.mean,
            'std': scaler.std
        },
        'vq_checkpoint_path': VQ_CHECKPOINT_PATH
    }
    
    torch.save(checkpoint, checkpoint_path)
    print(f"   💾 Saved: {filename}")

print("✅ Checkpoint persistence functions defined")

## 🎨 Sample Generation Function

In [ ]:
def generate_samples(unet, vq_model, scheduler, scaler, device, num_samples=8, num_steps=50):
    """
    Generate samples using respaced DDPM sampling.
    
    Args:
        unet: Trained U-Net (will use EMA weights if available)
        vq_model: Frozen VQ decoder
        scheduler: Noise scheduler
        scaler: Latent scaler
        device: torch device
        num_samples: Number of samples to generate
        num_steps: Number of denoising steps (50 for respaced DDPM)
    
    Returns:
        Generated images tensor (num_samples, 3, 480, 640)
    """
    unet.eval()
    
    with torch.no_grad():
        # Start from pure noise
        z_t = torch.randn(num_samples, 4, 60, 80, device=device)
        
        # Respaced timesteps: uniform stride of 20
        timesteps = list(range(0, 1000, 20))[::-1]  # [980, 960, ..., 20, 0]
        
        # Iterative denoising
        for i, t in enumerate(timesteps):
            t_tensor = torch.tensor([t] * num_samples, device=device, dtype=torch.long)
            
            # Predict noise
            noise_pred = unet(z_t, t_tensor)
            
            # Get schedule parameters
            alpha_t = scheduler.alphas[t]
            alpha_cumprod_t = scheduler.alphas_cumprod[t]
            beta_t = scheduler.betas[t]
            
            # Compute mean
            mean = (1.0 / torch.sqrt(alpha_t)) * (
                z_t - (beta_t / torch.sqrt(1.0 - alpha_cumprod_t)) * noise_pred
            )
            
            # Add noise (except last step)
            if i < len(timesteps) - 1:
                noise = torch.randn_like(z_t)
                variance = beta_t
                z_t = mean + torch.sqrt(variance) * noise
            else:
                z_t = mean
        
        # z_t is now z_0 (denoised latent)
        z_0_scaled = z_t
        
        # Unscale
        z_0 = scaler.unscale(z_0_scaled)
        
        # Decode
        images = vq_model.decode(z_0)
        
        # Clamp to [0, 1]
        images = torch.clamp(images, 0, 1)
    
    unet.train()
    return images


def save_sample_grid(images, save_path, nrow=4):
    """
    Save a grid of generated images.
    
    Args:
        images: Tensor of images (N, 3, H, W)
        save_path: Path to save image
        nrow: Number of images per row
    """
    from torchvision.utils import make_grid
    
    grid = make_grid(images, nrow=nrow, padding=2, normalize=False)
    grid_np = grid.cpu().numpy().transpose(1, 2, 0)
    
    plt.figure(figsize=(12, 12))
    plt.imshow(grid_np)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

print("✅ Sample generation functions defined")

## 🚂 Training Loop

In [ ]:
import time
from tqdm import tqdm

print("="*70)
print(f"STARTING TRAINING ({TRAINING_MODE.upper()} MODE)")
print("="*70)
print(f"Max Epochs: {MAX_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Device: {device}")
print("="*70 + "\n")

start_time = time.time()

for epoch in range(training_state['epoch'], MAX_EPOCHS):
    epoch_start = time.time()
    
    # ========================================================================
    # TRAINING
    # ========================================================================
    unet.train()
    train_loss_epoch = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")
    for batch_idx, (images, _) in enumerate(pbar):
        images = images.to(device)
        
        # Encode to latents (frozen VQ encoder)
        with torch.no_grad():
            z_quantized, _, _ = vq_model.encode(images)
            z_0 = scaler.scale(z_quantized)
        
        # Sample timesteps and noise
        t = torch.randint(0, 1000, (images.shape[0],), device=device, dtype=torch.long)
        noise = torch.randn_like(z_0)
        
        # Forward diffusion
        z_t = scheduler.add_noise(z_0, t, noise)
        
        # Predict noise
        noise_pred = unet(z_t, t)
        
        # Compute loss
        loss = F.mse_loss(noise_pred, noise)
        
        # Check for NaN
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"\n⚠️  NaN/Inf loss detected at epoch {epoch+1}, batch {batch_idx}!")
            print(f"   Stopping training to prevent corruption.")
            break
        
        # Optimize
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        grad_norm = torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=GRAD_CLIP)
        
        optimizer.step()
        
        # Update EMA
        ema.update(unet)
        
        train_loss_epoch += loss.item()
        num_batches += 1
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'grad_norm': f'{grad_norm:.2f}'
        })
    
    avg_train_loss = train_loss_epoch / max(num_batches, 1)
    training_state['train_losses'].append(avg_train_loss)
    
    # ========================================================================
    # VALIDATION
    # ========================================================================
    unet.eval()
    val_loss_epoch = 0.0
    num_val_batches = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]  ")
        for images, _ in pbar:
            images = images.to(device)
            
            # Encode
            z_quantized, _, _ = vq_model.encode(images)
            z_0 = scaler.scale(z_quantized)
            
            # Sample timesteps and noise
            t = torch.randint(0, 1000, (images.shape[0],), device=device, dtype=torch.long)
            noise = torch.randn_like(z_0)
            
            # Forward diffusion
            z_t = scheduler.add_noise(z_0, t, noise)
            
            # Predict noise
            noise_pred = unet(z_t, t)
            
            # Compute loss
            loss = F.mse_loss(noise_pred, noise)
            
            val_loss_epoch += loss.item()
            num_val_batches += 1
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_val_loss = val_loss_epoch / max(num_val_batches, 1)
    training_state['val_losses'].append(avg_val_loss)
    
    epoch_time = time.time() - epoch_start
    
    # ========================================================================
    # LOGGING
    # ========================================================================
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS} Summary:")
    print(f"  Train Loss: {avg_train_loss:.6f}")
    print(f"  Val Loss:   {avg_val_loss:.6f}")
    print(f"  Time: {epoch_time/60:.1f} min")
    
    # ========================================================================
    # CHECKPOINTING
    # ========================================================================
    training_state['epoch'] = epoch + 1
    
    # Save periodic checkpoint
    if (epoch + 1) % SAVE_EVERY == 0:
        save_checkpoint(
            epoch=epoch+1,
            unet=unet,
            ema=ema,
            optimizer=optimizer,
            train_losses=training_state['train_losses'],
            val_losses=training_state['val_losses'],
            best_val_loss=training_state['best_val_loss'],
            checkpoint_dir=CHECKPOINT_DIR,
            filename=f"checkpoint_epoch_{epoch+1}.pth"
        )
        
        # Upload to Kaggle Dataset
        upload_checkpoint_to_kaggle_dataset(CHECKPOINT_DIR, CHECKPOINT_DATASET_NAME)
    
    # Save best model
    if avg_val_loss < training_state['best_val_loss']:
        print(f"  🎯 New best model! (prev: {training_state['best_val_loss']:.6f})")
        training_state['best_val_loss'] = avg_val_loss
        training_state['patience_counter'] = 0
        
        save_checkpoint(
            epoch=epoch+1,
            unet=unet,
            ema=ema,
            optimizer=optimizer,
            train_losses=training_state['train_losses'],
            val_losses=training_state['val_losses'],
            best_val_loss=training_state['best_val_loss'],
            checkpoint_dir=CHECKPOINT_DIR,
            filename="best_model.pth"
        )
        
        # Upload to Kaggle Dataset
        upload_checkpoint_to_kaggle_dataset(CHECKPOINT_DIR, CHECKPOINT_DATASET_NAME)
    else:
        training_state['patience_counter'] += 1
        print(f"  Patience: {training_state['patience_counter']}/{EARLY_STOPPING_PATIENCE}")
    
    # ========================================================================
    # SAMPLE GENERATION
    # ========================================================================
    if (epoch + 1) % SAMPLE_EVERY == 0:
        print(f"\n  🎨 Generating samples...")
        
        # Apply EMA weights for sampling
        backup = {name: param.clone() for name, param in unet.named_parameters()}
        ema.apply_shadow(unet)
        
        samples = generate_samples(
            unet=unet,
            vq_model=vq_model,
            scheduler=scheduler,
            scaler=scaler,
            device=device,
            num_samples=8,
            num_steps=50
        )
        
        # Restore training weights
        ema.restore(unet, backup)
        
        # Save sample grid
        sample_path = SAMPLES_DIR / f"samples_epoch_{epoch+1}.png"
        save_sample_grid(samples, sample_path, nrow=4)
        print(f"     Saved: {sample_path.name}")
    
    # ========================================================================
    # EARLY STOPPING
    # ========================================================================
    if training_state['patience_counter'] >= EARLY_STOPPING_PATIENCE:
        print(f"\n⏹️  Early stopping triggered (patience={EARLY_STOPPING_PATIENCE})")
        break
    
    print("="*70 + "\n")

total_time = time.time() - start_time

print("="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"Total Time: {total_time/3600:.2f} hours")
print(f"Best Val Loss: {training_state['best_val_loss']:.6f}")
print(f"Checkpoints saved to: {CHECKPOINT_DIR}")
print(f"Samples saved to: {SAMPLES_DIR}")
print("="*70)

## 📊 Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(training_state['train_losses'], label='Train Loss', linewidth=2)
plt.plot(training_state['val_losses'], label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('LDM Stage 2 Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

plot_path = OUTPUT_DIR / "training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print(f"✅ Training curves saved to {plot_path}")

## 🎨 Generate Final Samples

In [ ]:
print("Generating final sample grid...")

# Load best model
best_checkpoint = torch.load(CHECKPOINT_DIR / "best_model.pth", map_location=device)
unet.load_state_dict(best_checkpoint['model_state_dict'])

# Apply EMA weights
for name, param in unet.named_parameters():
    param.data.copy_(best_checkpoint['ema_state_dict'][name])

# Generate 16 samples
final_samples = generate_samples(
    unet=unet,
    vq_model=vq_model,
    scheduler=scheduler,
    scaler=scaler,
    device=device,
    num_samples=16,
    num_steps=50
)

# Save
final_sample_path = OUTPUT_DIR / "final_samples_16.png"
save_sample_grid(final_samples, final_sample_path, nrow=4)

print(f"✅ Final samples saved to {final_sample_path}")

# Display
from IPython.display import Image, display
display(Image(filename=str(final_sample_path)))

## 📥 Download Results

**On Kaggle, all outputs are in `/kaggle/working/` and can be downloaded from the Output tab.**

In [ ]:
print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)

print(f"\nCheckpoint Dataset: {CHECKPOINT_DATASET_NAME}")
print(f"  - Contains all checkpoints (survives session crashes)")
print(f"  - Can be attached to future sessions for resumption")

print(f"\nOutput Directory: {OUTPUT_DIR}")
print(f"  - Checkpoints: {len(list(CHECKPOINT_DIR.glob('*.pth')))} files")
print(f"  - Samples: {len(list(SAMPLES_DIR.glob('*.png')))} files")

print(f"\nBest Model:")
print(f"  - Val Loss: {training_state['best_val_loss']:.6f}")
print(f"  - Location: {CHECKPOINT_DIR / 'best_model.pth'}")

if IS_KAGGLE:
    print(f"\n📥 To download results:")
    print(f"  1. Go to the 'Output' tab in Kaggle")
    print(f"  2. Download the entire 'outputs' folder")
    print(f"  3. Or access checkpoints from Dataset: {CHECKPOINT_DATASET_NAME}")

print("\n" + "="*70)
print(f"🎉 LDM STAGE 2 {TRAINING_MODE.upper()} COMPLETE!")
print("="*70)